# 01 - Quantum State Data Exploration

Explore the synthetic quantum states and measurement data used for training.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from src.data.states import generate_random_states, generate_single_state
from src.data.measurements import compute_measurement_probabilities, simulate_measurements
from src.evaluation.metrics import purity, fidelity
from src.representation.constraints import check_dm_constraints

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Generate and Visualize Different State Types

In [ ]:
n_qubits = 2
d = 2**n_qubits

state_types = ['pure_haar', 'mixed_hs', 'mixed_ginibre', 'thermal', 'product']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for i, stype in enumerate(state_types):
    rho = generate_single_state(n_qubits, stype, seed=42 + i)
    info = check_dm_constraints(rho)
    
    # Real part
    im0 = axes[0, i].imshow(np.real(rho), cmap='RdBu_r')
    axes[0, i].set_title(f'{stype}\nPurity={info["purity"]:.3f}, Rank={info["rank"]}')
    plt.colorbar(im0, ax=axes[0, i])
    
    # Imaginary part
    im1 = axes[1, i].imshow(np.imag(rho), cmap='RdBu_r')
    axes[1, i].set_title(f'Imag')
    plt.colorbar(im1, ax=axes[1, i])

plt.suptitle(f'Density Matrices by State Type (n={n_qubits})', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Purity Distribution Across State Types

In [ ]:
n_samples = 2000
state_types_config = {
    'pure_haar': 0.40, 'mixed_hs': 0.30,
    'mixed_ginibre': 0.10, 'thermal': 0.10, 'product': 0.10
}

rhos, labels = generate_random_states(2, n_samples, state_types_config, seed=42)
purities = [purity(r) for r in rhos]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(purities, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(1.0/d, color='red', linestyle='--', label=f'Maximally mixed (1/d={1/d:.3f})')
axes[0].axvline(1.0, color='green', linestyle='--', label='Pure state')
axes[0].set_xlabel('Purity')
axes[0].set_ylabel('Count')
axes[0].set_title('Purity Distribution (All States)')
axes[0].legend()

# By type
type_purities = {}
for label, pur in zip(labels, purities):
    type_purities.setdefault(label, []).append(pur)

positions = range(len(type_purities))
bp = axes[1].boxplot([type_purities[t] for t in state_types], 
                      positions=positions, labels=state_types)
axes[1].set_ylabel('Purity')
axes[1].set_title('Purity by State Type')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Print statistics
for stype in state_types:
    vals = type_purities[stype]
    print(f'{stype:20s}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}, n={len(vals)}')

## 3. Measurement Probability Distributions

In [ ]:
# Compare exact probabilities vs simulated frequencies
n_qubits = 1
rho = generate_single_state(n_qubits, 'pure_haar', seed=42)

probs_exact = compute_measurement_probabilities(rho)
probs_100 = simulate_measurements(rho, n_shots=100, seed=1)
probs_1000 = simulate_measurements(rho, n_shots=1000, seed=1)
probs_10000 = simulate_measurements(rho, n_shots=10000, seed=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (freqs, title) in zip(axes, [
    (probs_100, '100 shots'), (probs_1000, '1000 shots'), (probs_10000, '10000 shots')
]):
    n_outcomes = len(probs_exact)
    x = np.arange(n_outcomes)
    width = 0.35
    ax.bar(x - width/2, probs_exact, width, label='Exact', alpha=0.7)
    ax.bar(x + width/2, freqs, width, label='Simulated', alpha=0.7)
    ax.set_xlabel('Outcome index')
    ax.set_ylabel('Probability')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Exact vs Simulated Measurement Probabilities (1 qubit)')
plt.tight_layout()
plt.show()

# KL divergence vs shot count
shot_levels = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
kl_divs = []
for n_shots in shot_levels:
    freqs = simulate_measurements(rho, n_shots=n_shots, seed=1)
    # KL divergence: sum p * log(p/q), with small epsilon
    eps = 1e-10
    kl = np.sum(probs_exact * np.log((probs_exact + eps) / (freqs + eps)))
    kl_divs.append(kl)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(shot_levels, kl_divs, 'o-', markersize=6)
ax.set_xlabel('Number of shots')
ax.set_ylabel('KL Divergence')
ax.set_title('KL Divergence vs Shot Count')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Dataset Statistics

In [ ]:
from src.data.dataset import QSTDataset

for n_qubits in [1, 2]:
    print(f'\n=== n_qubits = {n_qubits} ===')
    ds = QSTDataset(n_qubits=n_qubits, n_states=1000, n_shots=10000, seed=42)
    
    print(f'  States: {len(ds)}')
    print(f'  Cholesky dim: {ds.cholesky_dim}')
    print(f'  Condition dim: {len(ds[0]["condition"])}')
    print(f'  Density matrix shape: {ds[0]["density_matrix"].shape}')
    
    # State type distribution
    type_counts = Counter(ds.state_labels)
    for stype, count in sorted(type_counts.items()):
        print(f'    {stype}: {count} ({100*count/len(ds):.1f}%)')
    
    # Cholesky vector statistics
    cholesky_data = ds.cholesky_vectors
    print(f'  Cholesky: mean={np.mean(cholesky_data):.4f}, std={np.std(cholesky_data):.4f}')
    print(f'             min={np.min(cholesky_data):.4f}, max={np.max(cholesky_data):.4f}')

## Summary

- Different state types span the full range of purity from 1/d to 1
- Measurement noise decreases as ~1/sqrt(N_shots)
- Cholesky vectors have roughly zero mean with variance depending on state type
- The measurement condition vector dimension grows as 6^n, manageable for n <= 3